# RAG Evaluation Frameworks

Tools specifically for evaluating Retrieval-Augmented Generation (RAG) systems —
covering the RAG Triad (context relevance, groundedness, answer relevance), retrieval
ranking, and production RAG monitoring.

## Tool Comparison

| Tool | Type | Best for |
|------|------|----------|
| **RAGAS** | Open-source Python | Full RAG Triad + context precision/recall in one library |
| **TruLens** | Open-source Python | RAG Triad with chain-of-thought reasoning explanations |
| **DeepEval FaithfulnessMetric** | Open-source Python | Claim-level groundedness via QAG — most reliable faithfulness scorer |
| **Arize Phoenix** | Open-source | RAG trace visualisation with per-chunk relevance scores |
| **Galileo** | Cloud | RAG monitoring with hallucination alerts in production |
| **Openlayer** | Cloud | Groundedness scoring integrated into CI/CD |

## RAGAS

Use when you want a single library that covers the full triad plus context
precision/recall. Best for **offline eval on a golden dataset** before deployment.

**Strengths:** Comprehensive metric coverage, actively maintained, dataset integration.  
**Limitations:** Requires ground-truth answers for some metrics (`context_recall`, `answer_correctness`).

In [ ]:
# pip install ragas datasets
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_relevancy,
    context_precision,
    context_recall,
)
from datasets import Dataset

eval_dataset = Dataset.from_list([
    {
        "question":     "What is SAP AI Core?",
        "answer":       "<agent output>",
        "contexts":     ["<retrieved chunk 1>", "<retrieved chunk 2>"],
        "ground_truth": "<expected answer>"
    }
])

results = evaluate(eval_dataset, metrics=[
    faithfulness,
    answer_relevancy,
    context_relevancy,
    context_precision,
    context_recall,
])
print(results)
# {'faithfulness': 0.85, 'answer_relevancy': 0.91, ...}

## TruLens

Use when you need **human-readable explanations alongside scores** — each metric
returns a chain-of-thought reasoning trace, not just a number.

**Strengths:** CoT explanations make failures debuggable; works with LangChain apps.  
**Limitations:** OpenAI dependency for the provider; less flexible metric customisation.

In [ ]:
# pip install trulens-eval
from trulens.apps.langchain import TruChain
from trulens.providers.openai import OpenAI as TruOpenAI
from trulens.core import Feedback

provider = TruOpenAI()

f_groundedness      = Feedback(provider.groundedness_measure_with_cot_reasons)
f_answer_relevance  = Feedback(provider.relevance_with_cot_reasons)
f_context_relevance = Feedback(provider.context_relevance_with_cot_reasons)

tru_recorder = TruChain(
    your_rag_chain,
    feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance]
)
# Each feedback returns a score + CoT explanation string

## DeepEval — FaithfulnessMetric

Use when **faithfulness / hallucination detection** is your primary concern.
Uses the QAG (Question-Answer Generation) pattern: extracts atomic claims from the
answer, then verifies each claim individually against retrieved context.

**Strengths:** QAG is more reliable than direct LLM scoring; composable with other
DeepEval metrics (`GEval`, `AnswerRelevancy`, `HallucinationMetric`).  
**Limitations:** Can be slow on long answers with many claims.

In [ ]:
# pip install deepeval
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualRelevancyMetric,
)
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input="What is SAP AI Core?",
    actual_output="<agent answer>",
    retrieval_context=["<chunk 1>", "<chunk 2>"]
)

faithfulness_metric = FaithfulnessMetric(threshold=0.8)
faithfulness_metric.measure(test_case)

print(faithfulness_metric.score)   # 0.0 – 1.0
print(faithfulness_metric.reason)  # claim-level breakdown

## Arize Phoenix

Use for **local, visual RAG debugging** — visualises the full span tree with per-chunk
relevance scores alongside latency and token counts. Single container, zero cloud dependency.

**Strengths:** Open-source, runs locally, excellent for trace-level debugging, native drift detection.  
**Limitations:** Not designed for large-scale automated eval; better as a debugging tool than a batch scorer.

In [ ]:
# pip install arize-phoenix openinference-instrumentation-langchain
import phoenix as px
from openinference.instrumentation.langchain import LangChainInstrumentor
from opentelemetry.sdk.trace import TracerProvider

# Launch local Phoenix UI at http://localhost:6006
session = px.launch_app()

tracer_provider = TracerProvider()
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

# All subsequent RAG calls are automatically traced + scored in the UI

## Galileo

Use for **production RAG monitoring** — alerts when hallucination rate spikes above
threshold on live traffic without manual trace inspection.

**Strengths:** Automated alerting, production-grade throughput, no sampling needed.  
**Limitations:** Cloud-only, paid; overkill for early-stage systems.

## Openlayer

Use when you need **groundedness scoring gated into CI/CD** — scores every PR
against a dataset and fails the pipeline if faithfulness drops below threshold.

**Strengths:** CI/CD-native; combines RAG eval with deployment gating.  
**Limitations:** Cloud-only.

## Quick Decision Guide

```
Offline eval on a golden dataset      → RAGAS
Need explanations alongside scores    → TruLens
Faithfulness / hallucination focus    → DeepEval FaithfulnessMetric
Local debugging of individual traces  → Arize Phoenix
Production monitoring / alerting      → Galileo
CI/CD gating on groundedness          → Openlayer
Already using DeepEval for agents     → DeepEval FaithfulnessMetric (consistent toolchain)
```

## References

- [RAGAS documentation](https://docs.ragas.io)
- [TruLens documentation](https://www.trulens.org/docs)
- [DeepEval documentation](https://docs.confident-ai.com)
- [Arize Phoenix documentation](https://docs.arize.com/phoenix)
- [Galileo documentation](https://docs.galileo.ai)
- [Openlayer documentation](https://docs.openlayer.com)